In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# %% 셀 0: 데이터 로드
import os, json, random, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import polars as pl
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

POS_DIR = "./data/8_telop_position"
STT_DIR = "./data/4_stt_results"
EMB_PATH = "./data/8_text_embeddings.pt"
GRID_W = 80
GRID_H = 80
EVAL_PER_CHANNEL = 5
SEED = 42
NUM_WORKERS = 32
FPS = 10
MAX_ACTIVE = 150
MAX_TEXT_LEN = 200
SCALAR_DIM = 16
SEG_SCALAR_DIM = 4

text2emb = torch.load(EMB_PATH, weights_only=True)
EMB_DIM = next(iter(text2emb.values())).shape[0]
ZERO_EMB = torch.zeros(EMB_DIM)
print(f"✅ 임베딩: {len(text2emb):,}개 dim={EMB_DIM}")


def stt_frames_to_segments(df_stt):
    rows = df_stt.sort("frame_num").to_dicts()
    segments = []
    cur_text = None
    cur_start_frame = None
    prev_frame = None
    for r in rows:
        t = r["stt_text"]
        f = int(r["frame_num"])
        if t != cur_text:
            if cur_text is not None and cur_text != "":
                segments.append({
                    "start": cur_start_frame / FPS,
                    "end": (prev_frame + 1) / FPS,
                    "text": cur_text.strip(),
                })
            cur_text = t
            cur_start_frame = f
        prev_frame = f
    if cur_text is not None and cur_text != "":
        segments.append({
            "start": cur_start_frame / FPS,
            "end": (prev_frame + 1) / FPS,
            "text": cur_text.strip(),
        })
    return segments


def load_one(args):
    channel, path = args
    with open(path, "r") as f:
        data = json.load(f)
    instances = data.get("instances", [])
    duration = data.get("duration", 0.1)
    if instances:
        duration = max(duration, max(inst["end_sec"] for inst in instances))
    video_name = data.get("video", "")
    file_id = os.path.basename(path)[:-5]
    inst_list = []
    for inst in instances:
        # 정합성: w/h 를 먼저 결정 (cx parity 와 독립) → x0 그 다음 결정 → x1 = x0 + w
        # round() banker's rounding 회피: math.floor(x + 0.5) 로 half-away-from-zero
        gx_f = float(inst["grid_x"]); gy_f = float(inst["grid_y"])
        gw_f = float(inst["grid_w"]); gh_f = float(inst["grid_h"])
        gw_int = int(np.clip(math.floor(gw_f + 0.5), 1, GRID_W))
        gh_int = int(np.clip(math.floor(gh_f + 0.5), 1, GRID_H))
        # x0, y0 도 동일 방식. x1 = x0 + w 가 GRID 넘지 않게 클립
        x0 = int(np.clip(math.floor(gx_f - gw_int / 2.0 + 0.5),
                          0, GRID_W - gw_int))
        y0 = int(np.clip(math.floor(gy_f - gh_int / 2.0 + 0.5),
                          0, GRID_H - gh_int))
        x1 = x0 + gw_int
        y1 = y0 + gh_int
        inst_list.append({
            "text": inst["text"],
            "text_len": len(inst["text"]),
            "start": inst["start_sec"],
            "end": inst["end_sec"],
            "x0": x0, "y0": y0, "x1": x1, "y1": y1,
            "w": gw_int, "h": gh_int,
        })
    stt_path = os.path.join(STT_DIR, channel, file_id + ".parquet")
    stt_segments = []
    if os.path.exists(stt_path):
        try:
            df_stt = pl.read_parquet(stt_path, glob=False)
            stt_segments = stt_frames_to_segments(df_stt)
        except:
            pass
    return {
        "channel": channel, "video_name": video_name, "file_id": file_id,
        "instances": inst_list, "stt_segments": stt_segments, "duration": duration,
    }


json_paths = []
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for fname in sorted(os.listdir(ch_dir)):
        if fname.endswith(".json"):
            json_paths.append((channel, os.path.join(ch_dir, fname)))

samples = []
channel_set = set()
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(load_one, args): args for args in json_paths}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="로드"):
        result = fut.result()
        channel_set.add(result["channel"])
        samples.append(result)

channels = sorted(channel_set)
channel2id = {ch: i for i, ch in enumerate(channels)}

rng = random.Random(SEED)
by_channel = {}
for s in samples:
    by_channel.setdefault(s["channel"], []).append(s)

train_samples = []
eval_samples = []
for ch, ch_samples in by_channel.items():
    ch_samples.sort(key=lambda s: s["file_id"])
    rng.shuffle(ch_samples)
    n_eval = min(EVAL_PER_CHANNEL, len(ch_samples))
    eval_samples.extend(ch_samples[:n_eval])
    train_samples.extend(ch_samples[n_eval:])

train_samples = [s for s in train_samples if len(s["instances"]) > 0]
eval_samples  = [s for s in eval_samples  if len(s["instances"]) > 0]


CHAR_VOCAB_SIZE = 4096                   # 고정 capacity (실제 unique는 ~1823, 여유 큼)
char2id: dict = {}                       # build_char_vocab(samples) 호출 시 채워짐


def build_char_vocab(samples):
    """모든 텔롭 텍스트에서 unique char 추출 → char2id 갱신 (CHAR_VOCAB_SIZE 안에 fit)."""
    global char2id
    chars = set()
    for s in samples:
        for inst in s["instances"]:
            chars.update(inst["text"])
    char2id = {c: i + 2 for i, c in enumerate(sorted(chars))}    # 0=pad, 1=unk
    used = len(char2id) + 2
    assert used <= CHAR_VOCAB_SIZE, f"vocab overflow: {used} > {CHAR_VOCAB_SIZE}"
    return used


def text_to_char_ids(text, max_len=MAX_TEXT_LEN):
    """글자 단위 인코딩, train vocab 사용. 미등록 글자 → unk(1)."""
    ids = [char2id.get(c, 1) for c in text[:max_len]]
    return ids + [0] * (max_len - len(ids))


used_vocab = build_char_vocab(samples)
print(f"✅ 채널: {len(channels)}개  train: {len(train_samples):,}  eval: {len(eval_samples):,}")
print(f"📝 char vocab: {len(char2id)} unique + pad/unk = {used_vocab} used / CHAR_VOCAB_SIZE={CHAR_VOCAB_SIZE} capacity, MAX_TEXT_LEN={MAX_TEXT_LEN}")


✅ 임베딩: 2,167,019개 dim=1024


로드: 100%|██████████| 66400/66400 [00:14<00:00, 4446.23it/s]


✅ 채널: 664개  train: 56,892  eval: 2,984
📝 char vocab: 1821 unique + pad/unk = 1823 used / CHAR_VOCAB_SIZE=4096 capacity, MAX_TEXT_LEN=200


In [2]:
# %% 셀 1: 채널 서브샘플링
rng_ch = random.Random(42)
sampled_channels = rng_ch.sample(channels, 67)
sampled_set = set(sampled_channels)

train_samples = [s for s in train_samples if s["channel"] in sampled_set]
eval_samples  = [s for s in eval_samples  if s["channel"] in sampled_set]
channels = sampled_channels
channel2id = {ch: i for i, ch in enumerate(channels)}

print(f"🔬 {len(channels)}개 채널  train: {len(train_samples):,}  eval: {len(eval_samples):,}")


🔬 67개 채널  train: 5,574  eval: 288


In [3]:
# %% 셀 1.5: 채널별 레이아웃 통계 (train set에서만)
ch_data = {ch: {"cx": [], "cy": [], "w": [], "h": []} for ch in channels}
for s in train_samples:
    ch = s["channel"]
    for inst in s["instances"]:
        ch_data[ch]["cx"].append((inst["x0"] + inst["x1"]) / 2.0 / GRID_W)
        ch_data[ch]["cy"].append((inst["y0"] + inst["y1"]) / 2.0 / GRID_H)
        ch_data[ch]["w"].append(inst["w"] / GRID_W)
        ch_data[ch]["h"].append(inst["h"] / GRID_H)

ch_stats = {}
for ch in channels:
    d = ch_data[ch]
    cxs = np.array(d["cx"]); cys = np.array(d["cy"])
    ws  = np.array(d["w"]);  hs  = np.array(d["h"])
    ch_stats[ch] = np.array([
        cxs.mean(), cys.mean(), ws.mean(), hs.mean(),
        cxs.std(),  cys.std(),  ws.std(),  hs.std(),
    ], dtype=np.float32)

print(f"📊 채널 통계: {len(ch_stats)}개")


📊 채널 통계: 67개


In [4]:
# %% 셀 2: Segments (STT 경계 포함) + 세그먼트-인스턴스 retrieval 인덱스
from torch.utils.data import Dataset, DataLoader
import math

BATCH_SIZE = 512
NUM_WORKERS_DL = 8
MAX_SEG_LOG = math.log(MAX_ACTIVE * 100)
RETRIEVAL_K = 4
RET_SCALAR_DIM = 8                       # text_len, ch_sim, vid_sim, stt_sim, has_stt, start, end, dur
RET_SCALAR_W = 0.5                       # lookup 시 scalar 가중치
# char vocab / text_to_char_ids / build_char_vocab 는 Cell 0 에서 정의됨


def make_segments(s):
    """인스턴스 경계 + STT 경계로 분할.
    각 segment는 0개 또는 1개의 STT를 가짐 (모호성 없음)."""
    insts = s["instances"]
    duration = max(s["duration"], 0.1)
    bounds = {0.0, duration}
    for inst in insts:
        bounds.add(inst["start"])
        bounds.add(inst["end"])
    for sg in s["stt_segments"]:
        bounds.add(sg["start"])
        bounds.add(sg["end"])
    bounds = sorted(b for b in bounds if 0 <= b <= duration)
    segments = []
    for t0, t1 in zip(bounds, bounds[1:]):
        if t1 - t0 <= 1e-4:
            continue
        mid = (t0 + t1) / 2
        active_idx = [j for j, inst in enumerate(insts)
                      if inst["start"] <= mid < inst["end"]]
        if not active_idx:
            continue
        if len(active_idx) > MAX_ACTIVE:
            active_idx = sorted(active_idx,
                                key=lambda j: -insts[j]["text_len"])[:MAX_ACTIVE]
        # 이 segment에 해당하는 STT 텍스트 (분할 후엔 정확히 0 또는 1개)
        stt_text_at_mid = None
        for sg in s["stt_segments"]:
            if sg["start"] <= mid < sg["end"]:
                stt_text_at_mid = sg["text"]
                break
        segments.append({
            "t0": t0, "t1": t1, "mid": mid,
            "active_idx": active_idx,
            "seg_len_frames": max(1.0, (t1 - t0) * FPS),
            "stt_text": stt_text_at_mid,
        })
    return segments


def build_segment_index(video_samples):
    entries = []
    for s in tqdm(video_samples, desc="segment 분해"):
        for seg in make_segments(s):
            entries.append((s, seg))
    return entries


train_entries = build_segment_index(train_samples)
eval_entries  = build_segment_index(eval_samples)
print(f"📦 train: {len(train_entries):,}  eval: {len(eval_entries):,} segments")

# ===== 채널별 (segment, instance) retrieval 인덱스 =====
# 각 entry: diff_ch + 8-dim scalar (segment-specific stt_sim 포함) + bbox + file_id
file_id2idx = {}
for s in train_samples:
    fid = s["file_id"]
    if fid not in file_id2idx:
        file_id2idx[fid] = len(file_id2idx)

_channel_emb_cache = {}
_video_emb_cache = {}
for ch in channels:
    e = text2emb.get(ch, ZERO_EMB)
    _channel_emb_cache[ch] = F.normalize(e.float(), dim=-1)

# Step 1: 모든 entry를 flat list로 수집 (Python overhead만)
print("📦 entries 수집...")
all_rows = []
for s, seg in tqdm(train_entries, desc="entries 수집", leave=False):
    fid_idx = file_id2idx[s["file_id"]]
    stt_text = seg.get("stt_text")
    video_dur = max(s["duration"], 0.1)
    for j in seg["active_idx"]:
        all_rows.append((
            s["channel"], fid_idx, s["video_name"], stt_text,
            s["instances"][j], video_dur,
        ))
print(f"   총 {len(all_rows):,} entries")

# Step 2: video / stt embedding cache (한 번씩만 normalize)
# train + eval 모두 포함 (eval 쿼리도 사전 계산에 필요)
print("🔧 video/stt 임베딩 캐시 (train + eval)...")
all_video_names = set(r[2] for r in all_rows)
all_stt_texts   = set(r[3] for r in all_rows if r[3])
for s, seg in eval_entries:
    all_video_names.add(s["video_name"])
    if seg.get("stt_text"):
        all_stt_texts.add(seg["stt_text"])

for v in all_video_names:
    if v not in _video_emb_cache:
        _video_emb_cache[v] = F.normalize(text2emb.get(v, ZERO_EMB).float(), dim=-1)
_stt_emb_cache = {}
for t in all_stt_texts:
    _stt_emb_cache[t] = F.normalize(text2emb.get(t, ZERO_EMB).float(), dim=-1)

# Step 3: 채널별로 그룹화
print("📂 채널별 그룹화...")
from collections import defaultdict
rows_by_channel = defaultdict(list)
for r in all_rows:
    rows_by_channel[r[0]].append(r)

# Step 4: 채널별 벡터화 빌드
print("🚀 벡터화 인덱스 빌드...")
ch_index = {}
ZERO_VEC_D = torch.zeros(EMB_DIM)
for ch in tqdm(channels, desc="채널 빌드"):
    rows = rows_by_channel.get(ch, [])
    M = len(rows)
    if M == 0:
        continue

    ch_emb_vec = _channel_emb_cache[ch]                          # [D]

    # 1) text embeddings (raw, then batch normalize)
    text_embs_raw = torch.stack([
        text2emb.get(r[4]["text"], ZERO_EMB).float()
        for r in rows
    ])                                                            # [M, D]
    text_embs = F.normalize(text_embs_raw, dim=-1)

    # 2) video embeddings (from cache)
    vid_embs = torch.stack([_video_emb_cache[r[2]] for r in rows])    # [M, D]

    # 3) STT embeddings (zero for no-STT)
    has_stt_t = torch.tensor([1.0 if r[3] else 0.0 for r in rows])     # [M]
    stt_embs = torch.stack([
        _stt_emb_cache[r[3]] if r[3] else ZERO_VEC_D for r in rows
    ])                                                                  # [M, D]

    # 4) Vectorized cosine similarities (lookup용 scalar)
    ch_sim  = (text_embs * ch_emb_vec.unsqueeze(0)).sum(-1)            # [M]
    vid_sim = (text_embs * vid_embs).sum(-1)
    stt_sim = (text_embs * stt_embs).sum(-1) * has_stt_t                # zero for no-STT

    # 5) Scalars
    text_len_t = torch.tensor([r[4]["text_len"] / MAX_TEXT_LEN for r in rows])
    starts     = torch.tensor([r[4]["start"] / r[5] for r in rows])
    ends       = torch.tensor([r[4]["end"]   / r[5] for r in rows])
    durs       = torch.tensor([(r[4]["end"] - r[4]["start"]) / r[5] for r in rows])

    scalar = torch.stack([
        text_len_t, ch_sim, vid_sim, stt_sim, has_stt_t,
        starts, ends, durs,
    ], dim=-1).float()                                                  # [M, 8]

    # 6) Bbox
    bbox = torch.tensor([
        [(r[4]["x0"] + r[4]["x1"]) / 2.0 / GRID_W,
         (r[4]["y0"] + r[4]["y1"]) / 2.0 / GRID_H,
         r[4]["w"] / GRID_W,
         r[4]["h"] / GRID_H]
        for r in rows
    ], dtype=torch.float32)                                             # [M, 4]

    file_id_idx_t = torch.tensor([r[1] for r in rows], dtype=torch.long)

    # 7) Store RAW embeddings (diff 대신) — 모델 본체 inst_emb_mlp와 같은 공간
    ch_index[ch] = {
        "text_emb":    text_embs,                # [M, D]  normalized
        "video_emb":   vid_embs,                 # [M, D]  normalized
        "stt_emb":     stt_embs,                 # [M, D]  zero if no-STT
        "scalar":      scalar,                   # [M, 8]
        "bbox":        bbox,                     # [M, 4]
        "file_id_idx": file_id_idx_t,            # [M]
    }

n_entries = sum(d["text_emb"].shape[0] for d in ch_index.values())
print(f"🔍 (segment, inst) 인덱스: {len(ch_index)}채널, 총 {n_entries:,} entry, K={RETRIEVAL_K}")
print(f"   각 entry: text_emb[{EMB_DIM}] + video_emb[{EMB_DIM}] + stt_emb[{EMB_DIM}] + scalar[{RET_SCALAR_DIM}] + bbox[4]")
print(f"   lookup: 4-way (sim_text + sim_stt_mag + sim_vid_mag + sim_other) / 4")


# ===== Top-K 사전 계산 (GPU 사용) =====
# 학습 중 __getitem__마다 matmul하는 대신, 시작할 때 한 번에 모든
# (entry, active_pos)의 top-K 인덱스를 계산해두면 retrieval은 O(1) 인덱싱.
PRECOMPUTE_BLOCK = 2048
precompute_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ZERO_VEC_D = torch.zeros(EMB_DIM)


def _query_features_for_entries(ch_entries):
    """채널의 ch_entries로부터 쿼리 텐서들을 만든다.
    반환: (text_emb [N, D], scalar [N, 8], fid [N])
    """
    q_text, q_sc, q_fid = [], [], []
    for entry_idx, s, seg in ch_entries:
        ch_emb_vec = _channel_emb_cache[s["channel"]]
        vid_emb_vec = _video_emb_cache[s["video_name"]]
        stt_text = seg.get("stt_text")
        if stt_text:
            stt_emb_vec = _stt_emb_cache.get(stt_text, ZERO_VEC_D)
            has_stt = 1.0
        else:
            stt_emb_vec = ZERO_VEC_D
            has_stt = 0.0
        fid = file_id2idx.get(s["file_id"], -1)
        video_dur = max(s["duration"], 0.1)
        for j in seg["active_idx"]:
            inst = s["instances"][j]
            text_emb_raw = text2emb.get(inst["text"], ZERO_EMB).float()
            text_emb = F.normalize(text_emb_raw, dim=-1)
            ch_sim  = (text_emb * ch_emb_vec).sum().item()
            vid_sim = (text_emb * vid_emb_vec).sum().item()
            stt_sim = (text_emb * stt_emb_vec).sum().item() * has_stt
            sc = torch.tensor([
                inst["text_len"] / MAX_TEXT_LEN,
                ch_sim, vid_sim, stt_sim, has_stt,
                inst["start"] / video_dur,
                inst["end"]   / video_dur,
                (inst["end"] - inst["start"]) / video_dur,
            ], dtype=torch.float32)
            q_text.append(text_emb); q_sc.append(sc); q_fid.append(fid)
    if not q_text:
        return None
    return (torch.stack(q_text), torch.stack(q_sc),
            torch.tensor(q_fid, dtype=torch.long))


def precompute_topk(entries, K, block_size=PRECOMPUTE_BLOCK, device=precompute_device):
    """각 entry의 active 인스턴스마다 같은 채널 ch_index 안에서 top-K 인덱스 사전 계산.
    Returns:
        topk_indices: list[len(entries)] of [N_active, K] long
        topk_mask:    list[len(entries)] of [N_active, K] bool
    """
    by_channel = defaultdict(list)
    for entry_idx, (s, seg) in enumerate(entries):
        by_channel[s["channel"]].append((entry_idx, s, seg))

    topk_indices = [None] * len(entries)
    topk_mask    = [None] * len(entries)

    for ch in tqdm(channels, desc="topk precompute", leave=False):
        ch_entries = by_channel.get(ch, [])
        if not ch_entries or ch not in ch_index:
            continue
        idx = ch_index[ch]
        M = idx["text_emb"].shape[0]
        if M == 0:
            continue

        # 인덱스를 GPU로 (이 채널만)
        i_text   = idx["text_emb"].to(device)               # [M, D]
        i_scalar = idx["scalar"].to(device)                  # [M, 8]
        i_fid    = idx["file_id_idx"].to(device)
        # lookup용 other scalars: [text_len, has_stt, start, end, dur]
        i_other_n   = F.normalize(i_scalar[:, [0, 4, 5, 6, 7]], dim=-1)
        i_vid_sim_T = i_scalar[:, 2:3].T   # [1, M]
        i_stt_sim_T = i_scalar[:, 3:4].T

        # 쿼리 빌드
        feats = _query_features_for_entries(ch_entries)
        if feats is None:
            continue
        Q_text, Q_sc, Q_fid = feats
        Q_text = Q_text.to(device)
        Q_sc   = Q_sc.to(device)
        Q_fid  = Q_fid.to(device)
        Q_other_n = F.normalize(Q_sc[:, [0, 4, 5, 6, 7]], dim=-1)

        N_q = Q_text.shape[0]
        all_top_idx  = torch.zeros(N_q, K, dtype=torch.long)
        all_top_mask = torch.zeros(N_q, K, dtype=torch.bool)

        # Block 처리 (메모리)
        for b in range(0, N_q, block_size):
            e = min(b + block_size, N_q)
            # 4-way lookup
            sim_text    = Q_text[b:e]    @ i_text.T                         # 텍스트 유사도
            sim_vid_mag = -((Q_sc[b:e, 2:3] - i_vid_sim_T).abs())            # 영상 정렬도 비슷한가
            sim_stt_mag = -((Q_sc[b:e, 3:4] - i_stt_sim_T).abs())            # STT 정렬도 비슷한가
            sim_other   = Q_other_n[b:e] @ i_other_n.T                       # 나머지 scalar
            sims = (sim_text + sim_vid_mag + sim_stt_mag + sim_other) / 4.0

            same_fid = (Q_fid[b:e].unsqueeze(1) == i_fid.unsqueeze(0))
            sims = sims.masked_fill(same_fid, float("-inf"))

            k_eff = min(K, M)
            topk = sims.topk(k_eff, dim=-1)
            if k_eff < K:
                top_idx = F.pad(topk.indices, (0, K - k_eff))
                top_val = F.pad(topk.values,  (0, K - k_eff), value=float("-inf"))
            else:
                top_idx = topk.indices
                top_val = topk.values
            top_mk = torch.isfinite(top_val)
            all_top_idx[b:e]  = top_idx.cpu()
            all_top_mask[b:e] = top_mk.cpu()

        # entry-major 순서대로 cursor 증가로 분배
        cursor = 0
        for entry_idx, s, seg in ch_entries:
            n_active = len(seg["active_idx"])
            topk_indices[entry_idx] = all_top_idx[cursor:cursor + n_active]
            topk_mask[entry_idx]    = all_top_mask[cursor:cursor + n_active]
            cursor += n_active

        # GPU 메모리 해제
        del i_text, i_scalar, i_fid, i_other_n
        del Q_text, Q_sc, Q_fid, Q_other_n
        if device.type == "cuda":
            torch.cuda.empty_cache()

    return topk_indices, topk_mask


print(f"⏳ top-K 사전 계산 (device={precompute_device})...")
train_topk_idx, train_topk_mask = precompute_topk(train_entries, RETRIEVAL_K)
eval_topk_idx,  eval_topk_mask  = precompute_topk(eval_entries,  RETRIEVAL_K)
print(f"✅ 사전 계산 완료: train {sum(1 for x in train_topk_idx if x is not None):,}, "
      f"eval {sum(1 for x in eval_topk_idx if x is not None):,}")


class SegmentDataset(Dataset):
    def __init__(self, entries, channel2id, text2emb, ch_stats,
                 ch_index, file_id2idx, retrieval_k,
                 topk_indices, topk_mask):
        self.entries = entries
        self.channel2id = channel2id
        self.text2emb = text2emb
        self.ch_stats = ch_stats
        self.ch_index = ch_index
        self.file_id2idx = file_id2idx
        self.retrieval_k = retrieval_k
        self.topk_indices = topk_indices    # list[len(entries)] of [N_active, K]
        self.topk_mask    = topk_mask        # list[len(entries)] of [N_active, K]

    def __len__(self):
        return len(self.entries)

    def _retrieve(self, channel, q_diff_ch_n, q_diff_vid_n, q_diff_stt_n,
                   q_scalar, self_fid_idx):
        """7-way 유사도 평균:
          (1) sim_ch_dir   = cos(diff_ch)            방향
          (2) sim_ch_mag   = -|ch_sim_q  - ch_sim_i|  크기
          (3) sim_vid_dir, (4) sim_vid_mag, (5) sim_stt_dir, (6) sim_stt_mag
          (7) sim_other    = cos([text_len, has_stt, start, end, dur])

        q_scalar: [N, 8] (inst_scalars[:, 0:8] 순서)
          0: text_len_norm  1: ch_sim  2: vid_sim  3: stt_sim
          4: has_stt        5: start_norm  6: end_norm  7: dur_norm
        """
        N, D = q_diff_ch_n.shape
        S = q_scalar.shape[1]
        K = self.retrieval_k

        idx = self.ch_index.get(channel)
        if idx is None or idx["emb_ch"].shape[0] == 0:
            return (
                torch.zeros(N, K, D), torch.zeros(N, K, D), torch.zeros(N, K, D),
                torch.zeros(N, K, 4), torch.zeros(N, K, S),
                torch.zeros(N, K, dtype=torch.bool),
            )

        i_diff_ch  = idx["emb_ch"]
        i_diff_vid = idx["emb_vid"]
        i_diff_stt = idx["emb_stt"]
        i_scalar   = idx["scalar"]
        i_bbox     = idx["bbox"]
        i_fid      = idx["file_id_idx"]
        M = i_diff_ch.shape[0]

        # === 방향 유사도 (3) ===
        sim_ch_dir  = q_diff_ch_n  @ i_diff_ch.T            # [N, M]
        sim_vid_dir = q_diff_vid_n @ i_diff_vid.T
        sim_stt_dir = q_diff_stt_n @ i_diff_stt.T

        # === 크기 유사도 (3): -|x_q - x_i| ===
        # q_scalar[:, 1] = ch_sim, [:, 2] = vid_sim, [:, 3] = stt_sim
        q_ch_sim  = q_scalar[:, 1:2]                        # [N, 1]
        q_vid_sim = q_scalar[:, 2:3]
        q_stt_sim = q_scalar[:, 3:4]
        i_ch_sim  = i_scalar[:, 1:2]                        # [M, 1]
        i_vid_sim = i_scalar[:, 2:3]
        i_stt_sim = i_scalar[:, 3:4]
        sim_ch_mag  = -((q_ch_sim  - i_ch_sim.T).abs())     # [N, M]
        sim_vid_mag = -((q_vid_sim - i_vid_sim.T).abs())
        sim_stt_mag = -((q_stt_sim - i_stt_sim.T).abs())

        # === Other features (1): [text_len, has_stt, start, end, dur] cosine ===
        # 인덱스: [0, 4, 5, 6, 7]
        q_other = q_scalar[:, [0, 4, 5, 6, 7]]              # [N, 5]
        i_other = i_scalar[:, [0, 4, 5, 6, 7]]              # [M, 5]
        q_other_n = F.normalize(q_other, dim=-1)
        i_other_n = F.normalize(i_other, dim=-1)
        sim_other = q_other_n @ i_other_n.T                  # [N, M]

        # 평균
        sims = (sim_ch_dir  + sim_ch_mag
              + sim_vid_dir + sim_vid_mag
              + sim_stt_dir + sim_stt_mag
              + sim_other) / 7.0

        same_fid = (i_fid.view(1, M) == int(self_fid_idx))
        sims = sims.masked_fill(same_fid, float("-inf"))

        k_eff = min(K, M)
        topk = sims.topk(k_eff, dim=-1)
        top_idx = topk.indices

        ret_diff_ch  = i_diff_ch[top_idx]
        ret_diff_vid = i_diff_vid[top_idx]
        ret_diff_stt = i_diff_stt[top_idx]
        ret_scalar   = i_scalar[top_idx]
        ret_bbox     = i_bbox[top_idx]
        ret_mask     = torch.isfinite(topk.values)

        if k_eff < K:
            pad_d = torch.zeros(N, K - k_eff, D)
            pad_b = torch.zeros(N, K - k_eff, 4)
            pad_s = torch.zeros(N, K - k_eff, S)
            pad_m = torch.zeros(N, K - k_eff, dtype=torch.bool)
            ret_diff_ch  = torch.cat([ret_diff_ch,  pad_d], dim=1)
            ret_diff_vid = torch.cat([ret_diff_vid, pad_d], dim=1)
            ret_diff_stt = torch.cat([ret_diff_stt, pad_d], dim=1)
            ret_bbox     = torch.cat([ret_bbox,     pad_b], dim=1)
            ret_scalar   = torch.cat([ret_scalar,   pad_s], dim=1)
            ret_mask     = torch.cat([ret_mask,     pad_m], dim=1)
        return ret_diff_ch, ret_diff_vid, ret_diff_stt, ret_bbox, ret_scalar, ret_mask

    def __getitem__(self, idx):
        s, seg = self.entries[idx]
        insts = s["instances"]
        active_idx = seg["active_idx"]
        N = len(active_idx)
        duration = max(s["duration"], 0.1)
        mid = seg["mid"]
        ch_stat = self.ch_stats[s["channel"]]

        channel_emb = F.normalize(self.text2emb.get(s["channel"],    ZERO_EMB), dim=-1)
        video_emb   = F.normalize(self.text2emb.get(s["video_name"], ZERO_EMB), dim=-1)

        # segment-specific STT (분할된 segment마다 정확히 0 또는 1개)
        stt_text = seg.get("stt_text")
        if stt_text:
            stt_emb = F.normalize(self.text2emb.get(stt_text, ZERO_EMB), dim=-1)
            has_stt = 1.0
        else:
            stt_emb = ZERO_EMB
            has_stt = 0.0

        active_insts = [insts[j] for j in active_idx]
        inst_embs = torch.stack([
            F.normalize(self.text2emb.get(inst["text"], ZERO_EMB), dim=-1)
            for inst in active_insts
        ])
        diff_ch  = inst_embs - channel_emb.unsqueeze(0)
        diff_vid = inst_embs - video_emb.unsqueeze(0)
        diff_stt = inst_embs - stt_emb.unsqueeze(0)

        ch_sims  = F.cosine_similarity(inst_embs, channel_emb.unsqueeze(0), dim=-1).numpy()
        vid_sims = F.cosine_similarity(inst_embs, video_emb.unsqueeze(0),   dim=-1).numpy()
        stt_sims = F.cosine_similarity(inst_embs, stt_emb.unsqueeze(0),     dim=-1).numpy()

        inst_scalars = np.zeros((N, SCALAR_DIM), dtype=np.float32)
        inst_scalars[:, 0] = [inst["text_len"] / MAX_TEXT_LEN for inst in active_insts]
        inst_scalars[:, 1] = ch_sims
        inst_scalars[:, 2] = vid_sims
        inst_scalars[:, 3] = stt_sims
        inst_scalars[:, 4] = has_stt
        inst_scalars[:, 5] = [inst["start"] / duration for inst in active_insts]
        inst_scalars[:, 6] = [inst["end"]   / duration for inst in active_insts]
        inst_scalars[:, 7] = [(inst["end"] - inst["start"]) / duration for inst in active_insts]
        inst_scalars[:, 8:16] = ch_stat[None, :]

        seg_scalar = np.array([
            mid / duration,
            (seg["t1"] - seg["t0"]) / duration,
            min(N / MAX_ACTIVE, 1.0),
            np.log1p(seg["seg_len_frames"]) / MAX_SEG_LOG,
        ], dtype=np.float32)

        gt_bbox = np.zeros((N, 4), dtype=np.float32)
        for i, inst in enumerate(active_insts):
            # 로딩 시 이미 계산된 x0,y0,x1,y1 그대로 사용 (cx 와 일치 보장됨)
            gt_bbox[i] = [inst["x0"], inst["y0"], inst["x1"], inst["y1"]]

        # Char-level IDs (UTF-8 bytes): 각 인스턴스 텍스트의 문자 시퀀스
        char_ids = np.array(
            [text_to_char_ids(inst["text"]) for inst in active_insts],
            dtype=np.int64,
        )                                       # [N, MAX_TEXT_LEN]

        # RALF retrieval: 사전 계산된 top-K 인덱스로 lookup (raw embedding)
        ch_idx = self.ch_index[s["channel"]]
        top_idx  = self.topk_indices[idx]              # [N_active, K] long
        top_mask = self.topk_mask[idx]                 # [N_active, K] bool
        ret_text_emb  = ch_idx["text_emb"][top_idx]    # [N_active, K, D]
        ret_video_emb = ch_idx["video_emb"][top_idx]
        ret_stt_emb   = ch_idx["stt_emb"][top_idx]     # zero if no-STT
        ret_scalar    = ch_idx["scalar"][top_idx]      # [N_active, K, 8]
        ret_bbox      = ch_idx["bbox"][top_idx]        # [N_active, K, 4]
        ret_mask      = top_mask

        return {
            "channel_id":   self.channel2id[s["channel"]],
            "inst_embs":    inst_embs,
            "diff_ch":      diff_ch,
            "diff_vid":     diff_vid,
            "diff_stt":     diff_stt,
            "inst_scalars": torch.from_numpy(inst_scalars),
            "seg_scalar":   torch.from_numpy(seg_scalar),
            "channel_emb":  channel_emb,
            "video_emb":    video_emb,
            "stt_emb":      stt_emb,
            "gt_bbox":      torch.from_numpy(gt_bbox),
            "seg_len":      float(seg["seg_len_frames"]),
            "n_active":     N,
            "char_ids":     torch.from_numpy(char_ids),
            "ret_text_emb":  ret_text_emb,
            "ret_video_emb": ret_video_emb,
            "ret_stt_emb":   ret_stt_emb,
            "ret_bbox":      ret_bbox,
            "ret_scalar":    ret_scalar,
            "ret_mask":      ret_mask,
        }


def collate(batch):
    B = len(batch)
    max_n = max(b["n_active"] for b in batch)
    D = batch[0]["inst_embs"].shape[1]
    K = batch[0]["ret_text_emb"].shape[1]

    inst_embs = torch.zeros(B, max_n, D)
    diff_ch   = torch.zeros(B, max_n, D)
    diff_vid  = torch.zeros(B, max_n, D)
    diff_stt  = torch.zeros(B, max_n, D)
    inst_scalars = torch.zeros(B, max_n, SCALAR_DIM)
    gt_bbox   = torch.zeros(B, max_n, 4)
    inst_mask = torch.zeros(B, max_n, dtype=torch.bool)
    char_ids  = torch.zeros(B, max_n, MAX_TEXT_LEN, dtype=torch.long)

    seg_scalar = torch.zeros(B, SEG_SCALAR_DIM)
    channel_emb = torch.zeros(B, D)
    video_emb   = torch.zeros(B, D)
    stt_emb     = torch.zeros(B, D)
    channel_id  = torch.zeros(B, dtype=torch.long)
    seg_len     = torch.zeros(B)

    ret_text_emb  = torch.zeros(B, max_n, K, D)
    ret_video_emb = torch.zeros(B, max_n, K, D)
    ret_stt_emb   = torch.zeros(B, max_n, K, D)
    ret_bbox      = torch.zeros(B, max_n, K, 4)
    ret_scalar    = torch.zeros(B, max_n, K, RET_SCALAR_DIM)
    ret_mask      = torch.zeros(B, max_n, K, dtype=torch.bool)

    for bi, b in enumerate(batch):
        n = b["n_active"]
        inst_embs[bi, :n]    = b["inst_embs"]
        diff_ch[bi, :n]      = b["diff_ch"]
        diff_vid[bi, :n]     = b["diff_vid"]
        diff_stt[bi, :n]     = b["diff_stt"]
        inst_scalars[bi, :n] = b["inst_scalars"]
        gt_bbox[bi, :n]      = b["gt_bbox"]
        inst_mask[bi, :n]    = True
        char_ids[bi, :n]     = b["char_ids"]
        seg_scalar[bi]   = b["seg_scalar"]
        channel_emb[bi]  = b["channel_emb"]
        video_emb[bi]    = b["video_emb"]
        stt_emb[bi]      = b["stt_emb"]
        channel_id[bi]   = b["channel_id"]
        seg_len[bi]      = b["seg_len"]
        ret_text_emb[bi, :n]  = b["ret_text_emb"]
        ret_video_emb[bi, :n] = b["ret_video_emb"]
        ret_stt_emb[bi, :n]   = b["ret_stt_emb"]
        ret_bbox[bi, :n]      = b["ret_bbox"]
        ret_scalar[bi, :n]    = b["ret_scalar"]
        ret_mask[bi, :n]      = b["ret_mask"]

    return {
        "inst_embs":    inst_embs,
        "diff_ch":      diff_ch,
        "diff_vid":     diff_vid,
        "diff_stt":     diff_stt,
        "inst_scalars": inst_scalars,
        "gt_bbox":      gt_bbox,
        "inst_mask":    inst_mask,
        "char_ids":     char_ids,
        "seg_scalar":   seg_scalar,
        "channel_emb":  channel_emb,
        "video_emb":    video_emb,
        "stt_emb":      stt_emb,
        "channel_id":   channel_id,
        "seg_len":      seg_len,
        "ret_text_emb":  ret_text_emb,
        "ret_video_emb": ret_video_emb,
        "ret_stt_emb":   ret_stt_emb,
        "ret_bbox":      ret_bbox,
        "ret_scalar":    ret_scalar,
        "ret_mask":      ret_mask,
    }


train_ds = SegmentDataset(train_entries, channel2id, text2emb, ch_stats,
                          ch_index, file_id2idx, RETRIEVAL_K,
                          train_topk_idx, train_topk_mask)
eval_ds  = SegmentDataset(eval_entries,  channel2id, text2emb, ch_stats,
                          ch_index, file_id2idx, RETRIEVAL_K,
                          eval_topk_idx,  eval_topk_mask)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS_DL, collate_fn=collate,
                          pin_memory=True, drop_last=True,
                          persistent_workers=True)
eval_loader  = DataLoader(eval_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS_DL, collate_fn=collate,
                          pin_memory=True,
                          persistent_workers=True)
print(f"🚚 train: {len(train_loader):,}  eval: {len(eval_loader):,} batches (persistent_workers=True)")


segment 분해: 100%|██████████| 288/288 [00:00<00:00, 2404.87it/s]


📦 train: 367,855  eval: 18,520 segments
📦 entries 수집...


   총 1,960,002 entries
🔧 video/stt 임베딩 캐시 (train + eval)...
📂 채널별 그룹화...
🚀 벡터화 인덱스 빌드...


채널 빌드: 100%|██████████| 67/67 [00:16<00:00,  4.15it/s]


🔍 (segment, inst) 인덱스: 67채널, 총 1,960,002 entry, K=4
   각 entry: text_emb[1024] + video_emb[1024] + stt_emb[1024] + scalar[8] + bbox[4]
   lookup: 4-way (sim_text + sim_stt_mag + sim_vid_mag + sim_other) / 4
⏳ top-K 사전 계산 (device=cuda)...


✅ 사전 계산 완료: train 367,855, eval 18,520
🚚 train: 718  eval: 37 batches (persistent_workers=True)


In [5]:
# %% 셀 3: COFS + spatial cross-attention
# DETR-Refine + 3-mode + Option 3 (cx-cy joint via spatial cross-attention)
D_MODEL = EMB_DIM
N_HEADS = 8
N_LAYERS = 6
FFN_DIM = D_MODEL * 4
DROPOUT = 0.1
N_CHANNELS = len(channels)
SPATIAL_D = 256   # spatial token dim (D_MODEL보다 작게 → 메모리)
# CHAR_VOCAB_SIZE, MAX_TEXT_LEN 은 cell 0 에서 정의됨 (글자 단위)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class CharCNN(nn.Module):
    """문자 시퀀스 → D-dim feature.
    Multi-scale Conv1d (k=3, 5, 7) + learnable-query attention pool.
    글자 단위 입력 (train vocab + pad/unk, CHAR_VOCAB_SIZE 안).
    Attention pool 이 글자 위치/종류/길이까지 weighted sum 으로 보존.
    """
    def __init__(self, vocab_size=CHAR_VOCAB_SIZE, char_dim=64,
                 hidden=128, output_dim=D_MODEL):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, char_dim, padding_idx=0)
        self.conv3 = nn.Conv1d(char_dim, hidden, kernel_size=3, padding=1)
        self.conv5 = nn.Conv1d(char_dim, hidden, kernel_size=5, padding=2)
        self.conv7 = nn.Conv1d(char_dim, hidden, kernel_size=7, padding=3)
        # 스케일별 learnable query (서로 다른 글자 패턴에 attend)
        scale = 1.0 / (hidden ** 0.5)
        self.pool_q3 = nn.Parameter(torch.randn(hidden) * scale)
        self.pool_q5 = nn.Parameter(torch.randn(hidden) * scale)
        self.pool_q7 = nn.Parameter(torch.randn(hidden) * scale)
        self.proj  = nn.Linear(hidden * 3, output_dim)

    def _attn_pool(self, feat, query, pad_mask, all_pad):
        """feat: [M, hidden, L], query: [hidden], pad_mask: [M, L] True=padding,
        all_pad: [M, 1] True 면 전체가 padding (빈 텍스트). Returns: [M, hidden]."""
        feat_t = feat.transpose(1, 2)                           # [M, L, hidden]
        logits = (feat_t * query).sum(dim=-1) / (feat.size(1) ** 0.5)
        logits = logits.masked_fill(pad_mask, float("-inf"))
        # 빈 텍스트 (전부 padding) 의 softmax NaN 방지: 0 으로 대체
        logits = torch.where(all_pad, torch.zeros_like(logits), logits)
        attn = F.softmax(logits, dim=-1)                        # [M, L]
        return (feat_t * attn.unsqueeze(-1)).sum(dim=1)         # [M, hidden]

    def forward(self, char_ids):
        """char_ids: [..., L] long. 임의 leading dim 허용 (B*N 등)."""
        orig_shape = char_ids.shape[:-1]
        L = char_ids.shape[-1]
        flat = char_ids.reshape(-1, L)                          # [M, L]
        pad_mask = (flat == 0)                                  # [M, L] True=padding
        all_pad = pad_mask.all(dim=-1, keepdim=True)            # [M, 1]
        x = self.embed(flat).transpose(1, 2)                    # [M, char_dim, L]
        h3 = F.gelu(self.conv3(x))                              # [M, hidden, L]
        h5 = F.gelu(self.conv5(x))
        h7 = F.gelu(self.conv7(x))
        p3 = self._attn_pool(h3, self.pool_q3, pad_mask, all_pad)
        p5 = self._attn_pool(h5, self.pool_q5, pad_mask, all_pad)
        p7 = self._attn_pool(h7, self.pool_q7, pad_mask, all_pad)
        h  = torch.cat([p3, p5, p7], dim=-1)                    # [M, 3*hidden]
        out = self.proj(h)                                       # [M, D]
        return out.reshape(*orig_shape, -1)                      # [..., D]


def build_2d_sinusoidal_pe(h, w, d):
    """2D sinusoidal positional encoding. d는 4의 배수여야 함.
    앞 d/2는 row 인코딩, 뒤 d/2는 col 인코딩."""
    assert d % 4 == 0, f"spatial dim {d} must be divisible by 4"
    d_half = d // 2
    div = torch.exp(torch.arange(0, d_half, 2).float()
                    * -(math.log(10000.0) / d_half))

    pe_row = torch.zeros(h, d_half)
    row_arg = torch.arange(h).float().unsqueeze(1) * div.unsqueeze(0)
    pe_row[:, 0::2] = torch.sin(row_arg)
    pe_row[:, 1::2] = torch.cos(row_arg)

    pe_col = torch.zeros(w, d_half)
    col_arg = torch.arange(w).float().unsqueeze(1) * div.unsqueeze(0)
    pe_col[:, 0::2] = torch.sin(col_arg)
    pe_col[:, 1::2] = torch.cos(col_arg)

    pe = torch.zeros(h, w, d)
    pe[:, :, :d_half] = pe_row.unsqueeze(1)        # broadcast over cols
    pe[:, :, d_half:] = pe_col.unsqueeze(0)        # broadcast over rows
    return pe


class CofsSpatialRalfLayoutModel(nn.Module):
    def __init__(self, emb_dim, n_channels,
                 d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS,
                 ffn=FFN_DIM, dropout=DROPOUT, spatial_d=SPATIAL_D):
        super().__init__()
        self.n_layers = n_layers
        self.spatial_d = spatial_d
        self.channel_embed = nn.Embedding(n_channels, d_model)

        # Context projection
        self.ctx_ch_proj     = nn.Linear(emb_dim, d_model)
        self.ctx_vid_proj    = nn.Linear(emb_dim, d_model)
        self.ctx_stt_proj    = nn.Linear(emb_dim, d_model)
        self.ctx_scalar_proj = nn.Linear(SEG_SCALAR_DIM, d_model)

        # Instance projection
        # MLP: 4개 raw 임베딩 (텔롭텍스트, 채널명, 영상명, STT) → D
        # diff/Hadamard 등 hand-crafted 없이, 모델이 임베딩 간 관계 직접 학습
        self.inst_emb_mlp = nn.Sequential(
            nn.Linear(4 * emb_dim, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

        # Reduced scalar path: text_len/sims/has_stt 제거, 임베딩으로 못 잡는 것만
        # = [start_norm, end_norm, dur_norm, ch_stats[8]] = 11 dim
        self.inst_scalar_kept = 11   # start + end + dur + ch_stats[8]
        self.inst_scalar_proj = nn.Linear(self.inst_scalar_kept, d_model)

        # Position uncertainty: retrieved bbox 분산 → "고정/랜덤" 신호
        # cx, cy std (2-dim) → D 차원 feature
        self.var_proj = nn.Linear(2, d_model)

        # CharCNN: 의미가 아닌 문자 구성 (특수문자, 괄호, 길이 패턴 등) 직접 학습
        self.char_cnn = CharCNN(
            vocab_size=CHAR_VOCAB_SIZE,
            char_dim=64, hidden=128, output_dim=d_model,
        )

        self.type_embed = nn.Embedding(2, d_model)

        # 6 transformer layers (self-attention)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=d_model, nhead=n_heads, dim_feedforward=ffn,
                dropout=dropout, activation="gelu", batch_first=True,
                norm_first=True,
            ) for _ in range(n_layers)
        ])
        self.head_norm = nn.LayerNorm(d_model)

        # === Spatial cross-attention for cx-cy joint ===
        # 1) Frozen 2D sinusoidal PE
        pe = build_2d_sinusoidal_pe(GRID_H, GRID_W, spatial_d)
        self.register_buffer("spatial_pe", pe)             # [80, 80, sd]
        # 2) Learnable spatial residual (zero-init)
        self.spatial_learn = nn.Parameter(torch.zeros(GRID_H, GRID_W, spatial_d))
        # 3) Query projection: instance feature → spatial query space
        self.spatial_q_proj = nn.Linear(d_model, spatial_d)
        # 4) Per-channel joint xy bias on logits (zero-init)
        self.ch_xy_bias = nn.Embedding(n_channels, GRID_W * GRID_H)
        nn.init.zeros_(self.ch_xy_bias.weight)

        # === w, h: factored Linear + 채널 marginal bias ===
        # w_head, h_head: inst_norm 만 받음
        self.w_head = nn.Linear(d_model, GRID_W)
        self.h_head = nn.Linear(d_model, GRID_H)
        # text_len 전용 MLP: 1 scalar → GRID_W / GRID_H logit 직접 사상
        len_hidden = 64
        self.w_len_mlp = nn.Sequential(
            nn.Linear(1, len_hidden),
            nn.GELU(),
            nn.Linear(len_hidden, GRID_W),
        )
        self.h_len_mlp = nn.Sequential(
            nn.Linear(1, len_hidden),
            nn.GELU(),
            nn.Linear(len_hidden, GRID_H),
        )
        nn.init.zeros_(self.w_head.bias)
        nn.init.zeros_(self.h_head.bias)
        self.ch_bias_w = nn.Embedding(n_channels, GRID_W)
        self.ch_bias_h = nn.Embedding(n_channels, GRID_H)
        nn.init.zeros_(self.ch_bias_w.weight)
        nn.init.zeros_(self.ch_bias_h.weight)

        # Reference encoder: bbox (4-dim) → D
        # 1st layer 는 기본 Kaiming init (정보 흐름) / 2nd layer 만 zero-init
        # (초기 출력 0 → 학습 초반 baseline 보존, 동시에 gradient 양쪽으로 흐름)
        self.ref_encoder = nn.Sequential(
            nn.Linear(4, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )
        nn.init.zeros_(self.ref_encoder[-1].weight)
        nn.init.zeros_(self.ref_encoder[-1].bias)

        # === RALF retrieval cross-attention ===
        # inst_emb_mlp를 query/key 양쪽에서 공유 (4 raw embeddings → D)
        # 별도 diff projection 없음 — query와 retrieved 모두 같은 MLP 거침
        self.ret_bbox_proj   = nn.Linear(4, d_model)
        self.ret_scalar_proj = nn.Linear(8, d_model)
        self.ret_norm = nn.LayerNorm(d_model)
        # ret_bbox_proj/ret_text_proj는 일반 init 그대로 (정보가 흘러야 함)
        # 하지만 inst_tok 잔차 추가 시 안정성을 위해 ret_norm 출력에 곱하는 gain 추가
        self.ret_gain = nn.Parameter(torch.zeros(1))    # zero-init → 초기엔 영향 없음

        # init_ref_proj 잔차 보정 (retrieved bbox weighted avg가 primary init_ref)
        self.init_ref_proj = nn.Linear(d_model, 4)
        nn.init.zeros_(self.init_ref_proj.weight)
        nn.init.zeros_(self.init_ref_proj.bias)

        # === Other-instance cross-attention (per refinement step) ===
        # 각 인스턴스가 자기 외 다른 인스턴스의 (ref, conf)를 명시적으로 참조.
        # query = inst_out, key/value = Linear([other_ref, other_conf]).
        # detach로 gradient cycle 방지, self_attn_mask로 자기 자신 제외.
        self.other_kv_proj = nn.ModuleList([
            nn.Linear(5, d_model) for _ in range(n_layers - 1)
        ])
        self.other_attn = nn.ModuleList([
            nn.MultiheadAttention(d_model, n_heads, batch_first=True, dropout=dropout)
            for _ in range(n_layers - 1)
        ])
        self.other_gain = nn.Parameter(torch.zeros(1))  # zero-init 안전장치

        # COFS 3-mode (training only)
        self.co_prob = 0.2
        self.dn_prob = 0.3
        self.dn_noise_scale = 0.1
        self.context_flag = nn.Embedding(2, d_model)
        nn.init.zeros_(self.context_flag.weight)

        # softargmax indices
        self.register_buffer("_x_arange", torch.arange(GRID_W).float())
        self.register_buffer("_y_arange", torch.arange(GRID_H).float())

    def _spatial_flat(self):
        """Return spatial token map flattened: [6400, spatial_d]."""
        sp = self.spatial_pe + self.spatial_learn       # [H, W, sd]
        return sp.view(GRID_H * GRID_W, self.spatial_d)

    def _heads(self, inst_norm, channel_id, text_len):
        """Return (xy_logits[B,N,6400], w_logits[B,N,80], h_logits[B,N,80]).
        text_len: [B, N, 1] 정규화 글자수."""
        q = self.spatial_q_proj(inst_norm)              # [B, N, sd]
        spatial_flat = self._spatial_flat()             # [6400, sd]
        # Dot product attention (single head)
        scale = float(self.spatial_d) ** 0.5
        xy_logits = torch.einsum("bnd, sd -> bns", q.float(),
                                  spatial_flat.float()) / scale
        # Per-channel joint xy bias
        xy_logits = xy_logits + self.ch_xy_bias(channel_id).unsqueeze(1).float()

        # w, h: inst_norm + text_len 전용 MLP logit 합산
        b_w = self.ch_bias_w(channel_id).unsqueeze(1)
        b_h = self.ch_bias_h(channel_id).unsqueeze(1)
        w_logits = self.w_head(inst_norm) + self.w_len_mlp(text_len) + b_w
        h_logits = self.h_head(inst_norm) + self.h_len_mlp(text_len) + b_h
        return xy_logits, w_logits, h_logits

    def _soft_decode(self, xy_logits, w_logits, h_logits):
        """xy_logits: [B, N, 6400] joint → cx, cy soft expectation."""
        B, N, _ = xy_logits.shape
        p_xy = F.softmax(xy_logits.float(), dim=-1).view(B, N, GRID_H, GRID_W)
        # marginalize
        p_cx = p_xy.sum(dim=-2)                          # [B, N, W]
        p_cy = p_xy.sum(dim=-1)                          # [B, N, H]
        cx = (p_cx * self._x_arange).sum(-1) / float(GRID_W)
        cy = (p_cy * self._y_arange).sum(-1) / float(GRID_H)

        p_w = F.softmax(w_logits.float(), dim=-1)
        p_h = F.softmax(h_logits.float(), dim=-1)
        w = ((p_w * self._x_arange).sum(-1) + 1.0) / float(GRID_W)
        h = ((p_h * self._y_arange).sum(-1) + 1.0) / float(GRID_H)
        return torch.stack([cx, cy, w, h], dim=-1)

    def _confidence(self, xy_logits, w_logits, h_logits):
        """Joint xy peak * w peak * h peak."""
        pk_xy = F.softmax(xy_logits.float(), -1).max(-1).values
        pk_w  = F.softmax(w_logits.float(),  -1).max(-1).values
        pk_h  = F.softmax(h_logits.float(),  -1).max(-1).values
        return pk_xy * pk_w * pk_h

    def _argmax_decode(self, xy_logits, w_logits, h_logits):
        idx = xy_logits.float().argmax(-1)               # [B, N] in [0, 6400)
        cy = (idx // GRID_W).float()
        cx = (idx %  GRID_W).float()
        w  = (w_logits.float().argmax(-1) + 1).float()
        h  = (h_logits.float().argmax(-1) + 1).float()
        return torch.stack([cx, cy, w, h], dim=-1)

    def forward(self, batch):
        B = batch["inst_embs"].size(0)
        N = batch["inst_embs"].size(1)

        # MLP path: 4개 raw 임베딩 concat → MLP가 관계 직접 학습
        text    = batch["inst_embs"]                                # [B, N, D_emb]
        ch_e    = batch["channel_emb"].unsqueeze(1).expand_as(text) # [B, N, D_emb]
        vid_e   = batch["video_emb"].unsqueeze(1).expand_as(text)
        stt_e   = batch["stt_emb"].unsqueeze(1).expand_as(text)
        emb_4   = torch.cat([text, ch_e, vid_e, stt_e], dim=-1)     # [B, N, 4D]
        inst_tok_emb = self.inst_emb_mlp(emb_4)                      # [B, N, D]

        # Scalar path (축소): start, end, dur, ch_stats만 (inst_scalars[:, 5:])
        kept_scalars = batch["inst_scalars"][:, :, 5:]               # [B, N, 11] start/end/dur/ch_stats
        inst_tok_scalar = self.inst_scalar_proj(kept_scalars)        # [B, N, D]

        # text_len: inst_tok 에 안 섞고 w/h head 에만 직접 사용
        text_len = batch["inst_scalars"][:, :, 0:1]                  # [B, N, 1] 정규화 글자수

        # CharCNN path: surface 패턴
        char_feat = self.char_cnn(batch["char_ids"])                 # [B, N, D]

        # Position uncertainty: retrieved K개 bbox의 xy std → "고정/랜덤" 신호
        # 4-way retrieval이라 같은 컨텍스트 K개 → conditional std (K=4도 의미있음)
        ret_xy = batch["ret_bbox"][..., :2]                              # [B, N, K, 2]
        ret_mask_f = batch["ret_mask"].unsqueeze(-1).float()             # [B, N, K, 1]
        n_valid = ret_mask_f.sum(dim=2).clamp(min=1.0)                   # [B, N, 1]
        ret_xy_mean = (ret_xy * ret_mask_f).sum(dim=2) / n_valid         # [B, N, 2]
        ret_xy_diff = (ret_xy - ret_xy_mean.unsqueeze(2)) * ret_mask_f   # [B, N, K, 2]
        ret_xy_var = (ret_xy_diff ** 2).sum(dim=2) / n_valid             # [B, N, 2]
        ret_xy_std = (ret_xy_var + 1e-8).sqrt()                          # [B, N, 2]
        var_feat = self.var_proj(ret_xy_std)                             # [B, N, D]

        inst_tok = inst_tok_emb + inst_tok_scalar + char_feat + var_feat
        inst_tok = inst_tok + self.type_embed.weight[1]

        ch_id_emb = self.channel_embed(batch["channel_id"])
        ctx_tok = (self.ctx_ch_proj(batch["channel_emb"])
                   + self.ctx_vid_proj(batch["video_emb"])
                   + self.ctx_stt_proj(batch["stt_emb"])
                   + self.ctx_scalar_proj(batch["seg_scalar"])
                   + ch_id_emb)
        ctx_tok = ctx_tok + self.type_embed.weight[0]
        ctx_tok = ctx_tok.unsqueeze(1)

        # === RALF: per-instance cross-attention to K retrievals ===
        # Query/Key 양쪽이 같은 inst_emb_mlp 거침 (4 raw embeddings → D)
        # Query: 자기 자신의 4 emb (이미 위에서 emb_4_q로 만들었지만 재계산 — 코드 명료성)
        ch_e_q  = batch["channel_emb"].unsqueeze(1).expand_as(batch["inst_embs"])  # [B,N,D]
        vid_e_q = batch["video_emb"].unsqueeze(1).expand_as(batch["inst_embs"])
        stt_e_q = batch["stt_emb"].unsqueeze(1).expand_as(batch["inst_embs"])
        emb_4_q_full = torch.cat([
            batch["inst_embs"], ch_e_q, vid_e_q, stt_e_q,
        ], dim=-1)                                                              # [B, N, 4D]
        q_emb_proj = self.inst_emb_mlp(emb_4_q_full)                            # [B, N, D]

        # Retrieved: ret_4_emb는 retrieved의 4종 (channel_emb는 query와 같음)
        K_size = batch["ret_text_emb"].size(2)
        ch_e_k = batch["channel_emb"].view(B, 1, 1, -1).expand(B, N, K_size, -1)
        ret_4 = torch.cat([
            batch["ret_text_emb"],
            ch_e_k,
            batch["ret_video_emb"],
            batch["ret_stt_emb"],
        ], dim=-1)                                                              # [B, N, K, 4D]
        ret_emb_proj    = self.inst_emb_mlp(ret_4)                              # [B, N, K, D]
        ret_bbox_feat   = self.ret_bbox_proj(batch["ret_bbox"])
        ret_scalar_feat = self.ret_scalar_proj(batch["ret_scalar"])
        ret_feat = ret_emb_proj + ret_bbox_feat + ret_scalar_feat               # [B, N, K, D]

        scale = float(inst_tok.size(-1)) ** 0.5
        attn_logits = torch.einsum("bnd, bnkd -> bnk",
                                    q_emb_proj.float(), ret_feat.float()) / scale
        # 무효 retrieval (mask=False, file_id 중복으로 인한 부족분) 제외
        attn_logits = attn_logits.masked_fill(~batch["ret_mask"], float("-inf"))
        # 만약 모든 K가 무효(채널 데이터 부족)면 softmax NaN 방지: -inf만 있을 때 0으로
        all_invalid = (~batch["ret_mask"]).all(dim=-1, keepdim=True)  # [B, N, 1]
        attn_logits = torch.where(all_invalid, torch.zeros_like(attn_logits),
                                   attn_logits)
        attn = F.softmax(attn_logits, dim=-1)                     # [B, N, K]
        # 무효 위치에서는 attn=0 보장
        attn = attn * batch["ret_mask"].float()

        # Pooled retrieval feature 주입 (zero-init gain → 초기엔 영향 없음)
        ret_pooled = torch.einsum("bnk, bnkd -> bnd", attn, ret_feat.float())
        inst_tok = inst_tok + self.ret_gain * self.ret_norm(ret_pooled.to(inst_tok.dtype))

        # init_ref: retrieved bbox의 attention-weighted avg
        # 모든 retrieval이 무효면 0.5로 fallback (중앙)
        ret_bbox_avg = torch.einsum("bnk, bnkc -> bnc", attn, batch["ret_bbox"].float())
        any_valid = batch["ret_mask"].any(dim=-1, keepdim=True).float()
        ret_bbox_avg = ret_bbox_avg * any_valid + 0.5 * (1.0 - any_valid)
        ret_bbox_avg = ret_bbox_avg.clamp(0.0, 1.0)

        # init_ref_proj는 잔차 보정 (zero-init이라 초기엔 0)
        ref_residual = torch.tanh(self.init_ref_proj(inst_tok)) * 0.1
        inst_ref = (ret_bbox_avg + ref_residual).clamp(0.0, 1.0)

        co_mask = torch.zeros(B, N, dtype=torch.bool, device=inst_tok.device)
        dn_mask = torch.zeros(B, N, dtype=torch.bool, device=inst_tok.device)
        gt_norm = None
        if self.training and ("gt_bbox" in batch):
            gx0 = batch["gt_bbox"][..., 0].float()
            gy0 = batch["gt_bbox"][..., 1].float()
            gx1 = batch["gt_bbox"][..., 2].float()
            gy1 = batch["gt_bbox"][..., 3].float()
            gt_norm = torch.stack([
                ((gx0 + gx1) / 2.0) / float(GRID_W),
                ((gy0 + gy1) / 2.0) / float(GRID_H),
                ((gx1 - gx0).clamp(min=1.0)) / float(GRID_W),
                ((gy1 - gy0).clamp(min=1.0)) / float(GRID_H),
            ], dim=-1)
            rand = torch.rand(B, N, device=inst_tok.device)
            co_mask = (rand < self.co_prob) & batch["inst_mask"]
            dn_mask = ((rand >= self.co_prob) &
                       (rand < self.co_prob + self.dn_prob) &
                       batch["inst_mask"])
            noise = torch.randn_like(gt_norm) * self.dn_noise_scale
            noisy_ref = (gt_norm + noise).clamp(0.0, 1.0)

            init_ref = inst_ref
            init_ref = torch.where(dn_mask.unsqueeze(-1), noisy_ref, init_ref)
            init_ref = torch.where(co_mask.unsqueeze(-1), gt_norm,   init_ref)
            inst_tok = inst_tok + self.context_flag(co_mask.long())
        else:
            init_ref = inst_ref

        ref_feat_init = self.ref_encoder(init_ref)
        inst_tok = inst_tok + ref_feat_init

        tokens = torch.cat([ctx_tok, inst_tok], dim=1)
        ctx_pad = torch.zeros(B, 1, dtype=torch.bool, device=tokens.device)
        pad_mask = torch.cat([ctx_pad, ~batch["inst_mask"]], dim=1)

        # Other-instance cross-attention용 mask (한 번만 만들기)
        # attn_mask [N, N]: True 위치는 무시. 대각선=자기 자신 → 제외
        self_attn_mask = torch.eye(N, dtype=torch.bool, device=tokens.device)
        # key_padding_mask [B, N]: 패딩된 인스턴스 무시
        key_pad_mask = ~batch["inst_mask"]

        channel_id = batch["channel_id"]
        all_logits = []
        for k, layer in enumerate(self.layers):
            tokens = layer(tokens, src_key_padding_mask=pad_mask)
            inst_out = self.head_norm(tokens[:, 1:])

            xy_l, w_l, h_l = self._heads(inst_out, channel_id, text_len)
            all_logits.append((xy_l, w_l, h_l))

            if k < self.n_layers - 1:
                ref = self._soft_decode(xy_l, w_l, h_l)
                conf = self._confidence(xy_l, w_l, h_l)
                gate = conf.unsqueeze(-1)
                # Mode C: 자기 ref와 conf 둘 다 GT 기반으로 덮어쓰기
                # (다른 인스턴스의 key/value로도 GT가 노출되어야 효과 큼)
                ref_for_others  = ref
                conf_for_others = conf
                if self.training and gt_norm is not None:
                    ref_for_others = torch.where(
                        co_mask.unsqueeze(-1), gt_norm, ref)
                    conf_for_others = torch.where(
                        co_mask, torch.ones_like(conf), conf)
                    ref = ref_for_others
                    gate = torch.where(co_mask.unsqueeze(-1),
                                        torch.ones_like(gate), gate)
                # 기존 ref_feedback (자기 자신 위치)
                ref_feat = self.ref_encoder(ref) * gate
                inst_part = tokens[:, 1:] + ref_feat

                # === NEW: other-instance cross-attention ===
                # 다른 인스턴스의 (ref, conf)를 detach 후 key/value로 사용
                other_kv_input = torch.cat([
                    ref_for_others.detach(),
                    conf_for_others.detach().unsqueeze(-1),
                ], dim=-1)                                     # [B, N, 5]
                other_kv = self.other_kv_proj[k](other_kv_input)  # [B, N, D]
                cross_out, _ = self.other_attn[k](
                    query=inst_out,
                    key=other_kv,
                    value=other_kv,
                    attn_mask=self_attn_mask,
                    key_padding_mask=key_pad_mask,
                    need_weights=False,
                )
                # N=1이거나 키가 전부 mask된 query는 NaN → 0으로 처리
                cross_out = torch.nan_to_num(cross_out, nan=0.0)
                inst_part = inst_part + self.other_gain * cross_out

                tokens = torch.cat([tokens[:, :1], inst_part], dim=1)

        xy_f, w_f, h_f = all_logits[-1]
        params = self._argmax_decode(xy_f, w_f, h_f)
        return all_logits, params, co_mask, dn_mask


model = CofsSpatialRalfLayoutModel(EMB_DIM, N_CHANNELS).to(device)
n_params = sum(p.numel() for p in model.parameters())
spatial_params = (model.spatial_learn.numel()
                  + model.spatial_q_proj.weight.numel()
                  + model.ch_xy_bias.weight.numel())
print(f"🧠 모델 params: {n_params/1e6:.2f}M | D={D_MODEL} sd={SPATIAL_D} L={N_LAYERS} H={N_HEADS}")
print(f"   3-mode: co={model.co_prob} dn={model.dn_prob} normal={1-model.co_prob-model.dn_prob:.2f}")
print(f"   Spatial cross-attn: 2D sinusoidal PE + learnable spatial[{GRID_H}×{GRID_W}×{SPATIAL_D}]")
print(f"   cx-cy joint: dot product attention → 6400-way + ch_xy_bias")
print(f"   w, h: factored Linear + channel marginal bias")
print(f"   spatial-관련 params: {spatial_params/1e6:.2f}M")
print(f"   RALF retrieval: K={RETRIEVAL_K}, 채널별 검색, self file_id 제외")
print(f"   ret_gain (zero-init): {model.ret_gain.item():.4f} → 학습 진행 따라 retrieval 영향 증가")
print(f"   init_ref = retrieved bbox weighted avg + 0.1*tanh(init_ref_proj)")
print(f"   Other-instance cross-attn: {N_LAYERS-1} layers, query=inst, key/value=other (ref+conf)")
print(f"   other_gain (zero-init): {model.other_gain.item():.4f} → 학습 진행 따라 영향 증가")
print(f"   Inst MLP: 4 raw embeddings (telop, ch, vid, stt) → MLP({4*EMB_DIM} → {D_MODEL})")
print(f"   Inst scalar (reduced): [start, end, dur, ch_stats[8]] = 11 → Linear → D")
print(f"   CharCNN: char-level (vocab={CHAR_VOCAB_SIZE}, used={len(char2id)+2}), multi-scale k=3,5,7, attn-pool, max_len={MAX_TEXT_LEN}")


🧠 모델 params: 109.46M | D=1024 sd=256 L=6 H=8
   3-mode: co=0.2 dn=0.3 normal=0.50
   Spatial cross-attn: 2D sinusoidal PE + learnable spatial[80×80×256]
   cx-cy joint: dot product attention → 6400-way + ch_xy_bias
   w, h: factored Linear + channel marginal bias
   spatial-관련 params: 2.33M
   RALF retrieval: K=4, 채널별 검색, self file_id 제외
   ret_gain (zero-init): 0.0000 → 학습 진행 따라 retrieval 영향 증가
   init_ref = retrieved bbox weighted avg + 0.1*tanh(init_ref_proj)
   Other-instance cross-attn: 5 layers, query=inst, key/value=other (ref+conf)
   other_gain (zero-init): 0.0000 → 학습 진행 따라 영향 증가
   Inst MLP: 4 raw embeddings (telop, ch, vid, stt) → MLP(4096 → 1024)
   Inst scalar (reduced): [start, end, dur, ch_stats[8]] = 11 → Linear → D
   CharCNN: char-level (vocab=4096, used=1823), multi-scale k=3,5,7, attn-pool, max_len=200


In [ ]:
# %% 셀 4: 학습 (deep supervision, 2D Gaussian for xy, Mode C 제외)
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS = 50
LR = 1e-4
WEIGHT_DECAY = 0.01
XY_LOSS_WEIGHT   = 1.0
W_LOSS_WEIGHT    = 1.0
H_LOSS_WEIGHT    = 1.0
L1_LOSS_WEIGHT   = 5.0      # DETR 표준 (decoded bbox L1)
GIOU_LOSS_WEIGHT = 2.0      # DETR 표준 (decoded bbox GIoU)
LABEL_SMOOTH_SIGMA = 1.0
LAYER_WEIGHT_RATIO = 1.0
SAVE_DIR = "./model/8_layout_cofs_spatial_ralf"
os.makedirs(SAVE_DIR, exist_ok=True)

_x_idx_f = torch.arange(GRID_W, device=device, dtype=torch.float32)
_y_idx_f = torch.arange(GRID_H, device=device, dtype=torch.float32)


def _gaussian_ce(logits, gt_target_idx_f, idx_f, sigma):
    g = idx_f.view(1, 1, -1)
    gauss = torch.exp(-0.5 * ((g - gt_target_idx_f.unsqueeze(-1)) / sigma) ** 2)
    target = gauss / gauss.sum(-1, keepdim=True).clamp(min=1e-8)
    log_p = F.log_softmax(logits.float(), dim=-1)
    return -(target * log_p).sum(-1)


def _gaussian_ce_2d(xy_logits, gt_cx, gt_cy, sigma):
    """xy_logits: [B, N, 6400], gt_cx/cy: [B, N]. Returns [B, N] loss."""
    B, N = gt_cx.shape
    xx = _x_idx_f.view(1, 1, 1, -1)
    yy = _y_idx_f.view(1, 1, -1, 1)
    cx_e = gt_cx.unsqueeze(-1).unsqueeze(-1)
    cy_e = gt_cy.unsqueeze(-1).unsqueeze(-1)
    dist_sq = (xx - cx_e) ** 2 + (yy - cy_e) ** 2
    gauss = torch.exp(-dist_sq / (2.0 * sigma * sigma))
    gauss_flat = gauss.view(B, N, -1)
    target = gauss_flat / gauss_flat.sum(-1, keepdim=True).clamp(min=1e-8)
    log_p = F.log_softmax(xy_logits.float(), dim=-1)
    return -(target * log_p).sum(-1)


def _decoded_bbox(xy_logits, w_logits, h_logits):
    """미분 가능한 expected bbox (cx, cy, w_actual, h_actual) — L1/GIoU 용."""
    B, N, _ = xy_logits.shape
    p_xy = F.softmax(xy_logits.float(), dim=-1).view(B, N, GRID_H, GRID_W)
    p_y = p_xy.sum(dim=-1)                      # [B, N, H]
    p_x = p_xy.sum(dim=-2)                      # [B, N, W]
    pred_cx = (p_x * _x_idx_f.view(1, 1, -1)).sum(-1)
    pred_cy = (p_y * _y_idx_f.view(1, 1, -1)).sum(-1)
    p_w = F.softmax(w_logits.float(), dim=-1)
    p_h = F.softmax(h_logits.float(), dim=-1)
    pred_w = (p_w * _x_idx_f.view(1, 1, -1)).sum(-1) + 1.0   # cell idx → 실제 width
    pred_h = (p_h * _y_idx_f.view(1, 1, -1)).sum(-1) + 1.0
    return pred_cx, pred_cy, pred_w, pred_h


def _giou_loss(pred_xyxy, gt_xyxy):
    """Generalized IoU loss = 1 - GIoU, in [0, 2]."""
    px0, py0, px1, py1 = pred_xyxy.unbind(-1)
    gx0, gy0, gx1, gy1 = gt_xyxy.unbind(-1)
    inter_w = (torch.min(px1, gx1) - torch.max(px0, gx0)).clamp(min=0)
    inter_h = (torch.min(py1, gy1) - torch.max(py0, gy0)).clamp(min=0)
    inter = inter_w * inter_h
    pa = (px1 - px0).clamp(min=0) * (py1 - py0).clamp(min=0)
    ga = (gx1 - gx0).clamp(min=0) * (gy1 - gy0).clamp(min=0)
    union = (pa + ga - inter).clamp(min=1e-8)
    iou = inter / union
    ex0 = torch.min(px0, gx0); ey0 = torch.min(py0, gy0)
    ex1 = torch.max(px1, gx1); ey1 = torch.max(py1, gy1)
    enclose = ((ex1 - ex0).clamp(min=0) * (ey1 - ey0).clamp(min=0)).clamp(min=1e-8)
    giou = iou - (enclose - union) / enclose
    return 1.0 - giou


def per_layer_losses(xy_l, w_l, h_l, gt_bbox, sigma=LABEL_SMOOTH_SIGMA):
    gx0 = gt_bbox[..., 0].float(); gy0 = gt_bbox[..., 1].float()
    gx1 = gt_bbox[..., 2].float(); gy1 = gt_bbox[..., 3].float()
    gt_cx = (gx0 + gx1) / 2.0
    gt_cy = (gy0 + gy1) / 2.0
    gt_w_actual = (gx1 - gx0).clamp(min=1.0)
    gt_h_actual = (gy1 - gy0).clamp(min=1.0)
    gt_w_idx = gt_w_actual - 1.0
    gt_h_idx = gt_h_actual - 1.0

    # CE (Gaussian smoothed) — mode 찍기
    l_xy = _gaussian_ce_2d(xy_l, gt_cx, gt_cy, sigma)
    l_w  = _gaussian_ce(w_l, gt_w_idx, _x_idx_f, sigma)
    l_h  = _gaussian_ce(h_l, gt_h_idx, _y_idx_f, sigma)

    # Decoded bbox L1 + GIoU — mIoU 직접 push
    pcx, pcy, pw, ph = _decoded_bbox(xy_l, w_l, h_l)
    l_l1 = ((pcx - gt_cx).abs() + (pcy - gt_cy).abs()
            + (pw - gt_w_actual).abs() + (ph - gt_h_actual).abs()) / float(GRID_W)
    pred_xyxy = torch.stack([pcx - pw/2.0, pcy - ph/2.0,
                              pcx + pw/2.0, pcy + ph/2.0], dim=-1)
    gt_xyxy   = torch.stack([gx0, gy0, gx1, gy1], dim=-1)
    l_giou = _giou_loss(pred_xyxy, gt_xyxy)

    return l_xy, l_w, l_h, l_l1, l_giou


def aggregate_per_token(loss_tok, loss_mask, seg_len_w):
    if not loss_mask.any():
        return loss_tok.sum() * 0.0
    m = loss_mask.float()
    counts = m.sum(dim=1).clamp(min=1.0)
    seg_loss = (loss_tok * m).sum(dim=1) / counts
    has_target = (m.sum(dim=1) > 0).float()
    w_n = seg_len_w * has_target
    w_n = w_n / w_n.sum().clamp(min=1e-8)
    return (seg_loss * w_n).sum()


def compute_losses_deep(all_logits, gt_bbox, loss_mask, seg_len_w,
                        layer_weight_ratio=LAYER_WEIGHT_RATIO):
    L = len(all_logits)
    weights = [layer_weight_ratio ** (L - 1 - k) for k in range(L)]
    s = sum(weights)
    weights = [w / s for w in weights]

    total = {"xy": 0.0, "w": 0.0, "h": 0.0, "l1": 0.0, "giou": 0.0}
    for k, (xy_l, w_l, h_l) in enumerate(all_logits):
        l_xy, l_w, l_h, l_l1, l_giou = per_layer_losses(xy_l, w_l, h_l, gt_bbox)
        wk = weights[k]
        total["xy"]   = total["xy"]   + wk * aggregate_per_token(l_xy,   loss_mask, seg_len_w)
        total["w"]    = total["w"]    + wk * aggregate_per_token(l_w,    loss_mask, seg_len_w)
        total["h"]    = total["h"]    + wk * aggregate_per_token(l_h,    loss_mask, seg_len_w)
        total["l1"]   = total["l1"]   + wk * aggregate_per_token(l_l1,   loss_mask, seg_len_w)
        total["giou"] = total["giou"] + wk * aggregate_per_token(l_giou, loss_mask, seg_len_w)
    return total["xy"], total["w"], total["h"], total["l1"], total["giou"]


@torch.no_grad()
def compute_metrics(pred_params, gt_bbox, inst_mask, seg_len_w, xy_logits=None):
    if not inst_mask.any():
        return None
    pred = pred_params.float()
    gt   = gt_bbox.float()
    cx, cy, w, h = pred.unbind(-1)
    gx0, gy0, gx1, gy1 = gt.unbind(-1)
    px0 = cx - w/2; py0 = cy - h/2
    px1 = cx + w/2; py1 = cy + h/2
    ix0 = torch.max(px0, gx0); iy0 = torch.max(py0, gy0)
    ix1 = torch.min(px1, gx1); iy1 = torch.min(py1, gy1)
    inter = (ix1 - ix0).clamp(min=0) * (iy1 - iy0).clamp(min=0)
    pa = (px1 - px0).clamp(min=0) * (py1 - py0).clamp(min=0)
    ga = (gx1 - gx0).clamp(min=0) * (gy1 - gy0).clamp(min=0)
    iou = inter / (pa + ga - inter + 1e-8)
    gt_cx = (gx0 + gx1) / 2; gt_cy = (gy0 + gy1) / 2
    gt_w  = (gx1 - gx0).clamp(min=1.0); gt_h = (gy1 - gy0).clamp(min=1.0)

    m = inst_mask.float()
    counts = m.sum(dim=1).clamp(min=1.0)
    w_n = seg_len_w / seg_len_w.sum().clamp(min=1e-8)
    miou       = ((iou * m).sum(1) / counts * w_n).sum().item()
    iou_50     = (((iou > 0.5).float() * m).sum(1) / counts * w_n).sum().item()
    iou_75     = (((iou > 0.75).float() * m).sum(1) / counts * w_n).sum().item()
    err_cx     = (((cx - gt_cx).abs() * m).sum(1) / counts * w_n).sum().item()
    err_cy     = (((cy - gt_cy).abs() * m).sum(1) / counts * w_n).sum().item()
    err_w      = (((w  - gt_w).abs()  * m).sum(1) / counts * w_n).sum().item()
    err_h      = (((h  - gt_h).abs()  * m).sum(1) / counts * w_n).sum().item()
    out = {
        "mIoU": miou, "IoU@0.5": iou_50, "IoU@0.75": iou_75,
        "errCX": err_cx, "errCY": err_cy, "errW": err_w, "errH": err_h,
    }
    if xy_logits is not None:
        B, N, _ = xy_logits.shape
        prob_xy = F.softmax(xy_logits.float(), dim=-1)
        prob_xy_2d = prob_xy.view(B, N, GRID_H, GRID_W)
        p_cx = prob_xy_2d.sum(dim=-2)                       # marginal x
        p_cy = prob_xy_2d.sum(dim=-1)                       # marginal y
        peak_cx = p_cx.max(-1).values
        peak_cy = p_cy.max(-1).values
        idx = prob_xy.argmax(-1)
        pred_cx = (idx %  GRID_W).float()
        pred_cy = (idx // GRID_W).float()
        # half-integer GT center 도 nearest cell 로 정확히 매핑 (.long() floor 회피)
        gt_cx_idx = ((gx0 + gx1) / 2.0).round().clamp(0, GRID_W - 1).long().float()
        gt_cy_idx = ((gy0 + gy1) / 2.0).round().clamp(0, GRID_H - 1).long().float()
        top1_cx = (pred_cx == gt_cx_idx).float()
        top1_cy = (pred_cy == gt_cy_idx).float()
        out.update({
            "peak_x":  ((peak_cx * m).sum(1) / counts * w_n).sum().item(),
            "peak_y":  ((peak_cy * m).sum(1) / counts * w_n).sum().item(),
            "cx@1":    ((top1_cx * m).sum(1) / counts * w_n).sum().item(),
            "cy@1":    ((top1_cy * m).sum(1) / counts * w_n).sum().item(),
        })
    return out


@torch.no_grad()
def metrics_with_buckets(all_logits, gt_bbox, inst_mask, seg_len_w):
    layer_results = []
    for xy_l, w_l, h_l in all_logits:
        idx = xy_l.float().argmax(-1)
        cy = (idx // GRID_W).float()
        cx = (idx %  GRID_W).float()
        w  = (w_l.float().argmax(-1) + 1).float()
        h  = (h_l.float().argmax(-1) + 1).float()
        params = torch.stack([cx, cy, w, h], dim=-1)
        m_dict = compute_metrics(params, gt_bbox, inst_mask, seg_len_w,
                                  xy_logits=xy_l)
        if m_dict is None:
            layer_results.append(None); continue

        # Joint xy peak as confidence
        prob_xy = F.softmax(xy_l.float(), dim=-1)
        conf = prob_xy.max(-1).values                      # [B, N] joint peak

        gx0, gy0, gx1, gy1 = gt_bbox.float().unbind(-1)
        px0 = cx - w/2; py0 = cy - h/2
        px1 = cx + w/2; py1 = cy + h/2
        ix0 = torch.max(px0, gx0); iy0 = torch.max(py0, gy0)
        ix1 = torch.min(px1, gx1); iy1 = torch.min(py1, gy1)
        inter = (ix1 - ix0).clamp(min=0) * (iy1 - iy0).clamp(min=0)
        pa = (px1 - px0).clamp(min=0) * (py1 - py0).clamp(min=0)
        ga = (gx1 - gx0).clamp(min=0) * (gy1 - gy0).clamp(min=0)
        iou = inter / (pa + ga - inter + 1e-8)

        m = inst_mask.float()
        seg_w_exp = seg_len_w.unsqueeze(-1).expand_as(conf)
        m_total = (m * seg_w_exp).sum().clamp(min=1e-8)
        # Joint xy peak는 6400-way라 1D peak보다 작음 → bucket scale을 0.05로 좁힘
        # 그래야 의미있는 분포 관찰 가능
        for kk in range(10):
            lo = kk * 0.05
            hi = (kk + 1) * 0.05
            if kk < 9:
                b_mask = (conf >= lo) & (conf < hi)
            else:
                b_mask = (conf >= lo)
            bm = (b_mask & inst_mask).float() * seg_w_exp
            bm_sum = bm.sum().clamp(min=1e-8)
            mi   = (iou * bm).sum() / bm_sum
            frac = bm.sum() / m_total
            m_dict[f"miou[{lo:.2f}]"] = mi.item()
            m_dict[f"frac[{lo:.2f}]"] = frac.item()
        layer_results.append(m_dict)
    return layer_results


def batch_to_device(batch, device):
    return {k: (v.to(device) if isinstance(v, torch.Tensor) else v)
            for k, v in batch.items()}


optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_score = -1.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    tot = {"loss":0.0, "xy":0.0, "w":0.0, "h":0.0, "l1":0.0, "giou":0.0}
    mode_counts = {"co": 0, "dn": 0, "normal": 0, "active": 0}
    n_b = 0
    for batch in tqdm(train_loader, desc=f"[{epoch}/{EPOCHS}] train", leave=False):
        batch = batch_to_device(batch, device)
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            all_logits, params, co_mask, dn_mask = model(batch)
            loss_mask = batch["inst_mask"] & (~co_mask)
            loss_xy, loss_w, loss_h, loss_l1, loss_giou = compute_losses_deep(
                all_logits, batch["gt_bbox"], loss_mask, batch["seg_len"],
            )
            loss = (XY_LOSS_WEIGHT   * loss_xy +
                    W_LOSS_WEIGHT    * loss_w  +
                    H_LOSS_WEIGHT    * loss_h  +
                    L1_LOSS_WEIGHT   * loss_l1 +
                    GIOU_LOSS_WEIGHT * loss_giou)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tot["loss"] += loss.item()
        tot["xy"]   += loss_xy.item()
        tot["w"]    += loss_w.item()
        tot["h"]    += loss_h.item()
        tot["l1"]   += loss_l1.item()
        tot["giou"] += loss_giou.item()
        with torch.no_grad():
            active = batch["inst_mask"]
            mode_counts["active"] += active.sum().item()
            mode_counts["co"]     += co_mask.sum().item()
            mode_counts["dn"]     += dn_mask.sum().item()
            mode_counts["normal"] += (active & ~co_mask & ~dn_mask).sum().item()
        n_b += 1

    model.eval()
    eval_loss_sum = 0.0
    n_eb = 0
    all_layer_metrics = [[] for _ in range(N_LAYERS)]
    with torch.no_grad():
        for batch in eval_loader:
            batch = batch_to_device(batch, device)
            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                all_logits, params, co_mask, dn_mask = model(batch)
                loss_xy, loss_w, loss_h, loss_l1, loss_giou = compute_losses_deep(
                    all_logits, batch["gt_bbox"], batch["inst_mask"], batch["seg_len"],
                )
                eloss = (XY_LOSS_WEIGHT   * loss_xy +
                         W_LOSS_WEIGHT    * loss_w  +
                         H_LOSS_WEIGHT    * loss_h  +
                         L1_LOSS_WEIGHT   * loss_l1 +
                         GIOU_LOSS_WEIGHT * loss_giou)
            layer_metrics = metrics_with_buckets(all_logits, batch["gt_bbox"],
                                                  batch["inst_mask"], batch["seg_len"])
            for k, lm in enumerate(layer_metrics):
                if lm is not None:
                    all_layer_metrics[k].append((lm, batch["seg_len"].sum().item()))
            eval_loss_sum += eloss.item()
            n_eb += 1

    def agg_dicts(metric_list):
        tw = sum(w for _, w in metric_list) or 1.0
        keys = list(metric_list[0][0].keys())
        return {k: sum(d[k] * w for d, w in metric_list) / tw for k in keys}

    layer_aggs = [agg_dicts(lst) for lst in all_layer_metrics]
    final_agg  = layer_aggs[-1]

    scheduler.step()
    bands = [f"{kk*0.05:.2f}" for kk in range(10)]

    act = max(mode_counts["active"], 1)
    co_pct  = mode_counts["co"] / act * 100
    dn_pct  = mode_counts["dn"] / act * 100
    nor_pct = mode_counts["normal"] / act * 100

    msg = ("[{ep}/{ne}]  train={tl:.4f} (xy={xy:.3f} w={wl:.3f} h={hl:.3f} "
           "l1={l1:.3f} gi={gi:.3f})  "
           "eval={el:.4f}  mIoU={miou:.4f}  cx@1={c_x:.3f} cy@1={c_y:.3f}  "
           "IoU@0.5={i50:.3f}  errCX={ecx:.2f} errCY={ecy:.2f} errW={ew:.2f} errH={eh:.2f}  "
           "pkX={px:.3f} pkY={py:.3f}  ret_g={rg:.4f}  oth_g={og:.4f}  lr={lr:.2e}"
           ).format(ep=epoch, ne=EPOCHS,
                    tl=tot["loss"]/max(n_b,1),
                    xy=tot["xy"]/max(n_b,1),
                    wl=tot["w"]/max(n_b,1), hl=tot["h"]/max(n_b,1),
                    l1=tot["l1"]/max(n_b,1), gi=tot["giou"]/max(n_b,1),
                    el=eval_loss_sum/max(n_eb,1),
                    miou=final_agg["mIoU"],
                    c_x=final_agg.get("cx@1", 0.0), c_y=final_agg.get("cy@1", 0.0),
                    i50=final_agg["IoU@0.5"],
                    ecx=final_agg["errCX"], ecy=final_agg["errCY"],
                    ew=final_agg["errW"], eh=final_agg["errH"],
                    px=final_agg.get("peak_x", 0.0), py=final_agg.get("peak_y", 0.0),
                    rg=model.ret_gain.item(),
                    og=model.other_gain.item(),
                    lr=scheduler.get_last_lr()[0])
    print(msg)
    print(f"           modes: co={co_pct:.1f}%  dn={dn_pct:.1f}%  normal={nor_pct:.1f}%")

    layer_miou_str = " → ".join(f"L{k+1}:{a['mIoU']:.3f}" for k, a in enumerate(layer_aggs))
    print(f"           layers: {layer_miou_str}")

    for k, a in enumerate(layer_aggs):
        bucket_str = "  ".join(
            f"{b}:{a.get(f'miou[{b}]', 0.0):.2f}[{a.get(f'frac[{b}]', 0.0)*100:.0f}%]"
            for b in bands
        )
        print(f"           L{k+1}: {bucket_str}")
    agg = final_agg

    if agg["mIoU"] > best_score:
        best_score = agg["mIoU"]
        torch.save({"model": model.state_dict(), "epoch": epoch,
                    "mIoU": best_score},
                   os.path.join(SAVE_DIR, "best.pt"))
        print(f"   💾 best 갱신 (mIoU={best_score:.4f})")


[1/50]  train=14.5714 (xy=5.671 w=3.539 h=1.795 l1=0.286 gi=1.067)  eval=14.2442  mIoU=0.2911  cx@1=0.573 cy@1=0.215  IoU@0.5=0.330  errCX=4.15 errCY=8.02 errW=8.55 errH=0.77  pkX=0.292 pkY=0.189  ret_g=-0.0039  oth_g=-0.0002  lr=9.99e-05
           modes: co=20.0%  dn=29.9%  normal=50.0%
           layers: L1:0.244 → L2:0.274 → L3:0.287 → L4:0.293 → L5:0.294 → L6:0.291
           L1: 0.00:0.16[78%]  0.05:0.40[20%]  0.10:0.10[2%]  0.15:0.00[0%]  0.20:0.00[0%]  0.25:0.00[0%]  0.30:0.00[0%]  0.35:0.00[0%]  0.40:0.00[0%]  0.45:0.00[0%]
           L2: 0.00:0.15[62%]  0.05:0.35[32%]  0.10:0.38[5%]  0.15:0.11[0%]  0.20:0.00[0%]  0.25:0.00[0%]  0.30:0.00[0%]  0.35:0.00[0%]  0.40:0.00[0%]  0.45:0.00[0%]
           L3: 0.00:0.15[56%]  0.05:0.35[37%]  0.10:0.43[7%]  0.15:0.15[1%]  0.20:0.00[0%]  0.25:0.00[0%]  0.30:0.00[0%]  0.35:0.00[0%]  0.40:0.00[0%]  0.45:0.00[0%]
           L4: 0.00:0.16[52%]  0.05:0.33[39%]  0.10:0.44[8%]  0.15:0.15[1%]  0.20:0.00[0%]  0.25:0.00[0%]  0.30:0.00[0%]  0.35:0.

[2/50]  train=12.2375 (xy=4.873 w=3.114 h=1.645 l1=0.188 gi=0.832)  eval=13.6840  mIoU=0.3492  cx@1=0.577 cy@1=0.259  IoU@0.5=0.403  errCX=3.90 errCY=6.90 errW=7.77 errH=0.71  pkX=0.294 pkY=0.229  ret_g=-0.0134  oth_g=-0.0004  lr=9.96e-05
           modes: co=20.0%  dn=30.0%  normal=50.0%
           layers: L1:0.304 → L2:0.328 → L3:0.344 → L4:0.347 → L5:0.348 → L6:0.349
           L1: 0.00:0.16[63%]  0.05:0.42[31%]  0.10:0.46[6%]  0.15:0.20[1%]  0.20:0.00[0%]  0.25:0.00[0%]  0.30:0.00[0%]  0.35:0.00[0%]  0.40:0.00[0%]  0.45:0.00[0%]
           L2: 0.00:0.13[44%]  0.05:0.37[39%]  0.10:0.53[14%]  0.15:0.41[2%]  0.20:0.00[0%]  0.25:0.00[0%]  0.30:0.00[0%]  0.35:0.00[0%]  0.40:0.00[0%]  0.45:0.00[0%]
           L3: 0.00:0.11[41%]  0.05:0.37[40%]  0.10:0.57[16%]  0.15:0.43[3%]  0.20:0.07[0%]  0.25:0.00[0%]  0.30:0.00[0%]  0.35:0.00[0%]  0.40:0.00[0%]  0.45:0.00[0%]
           L4: 0.00:0.10[38%]  0.05:0.35[39%]  0.10:0.55[18%]  0.15:0.46[4%]  0.20:0.07[0%]  0.25:0.00[0%]  0.30:0.00[0%]  0.35

[3/50]  train=10.9420 (xy=4.438 w=2.836 h=1.581 l1=0.143 gi=0.685)  eval=13.8162  mIoU=0.3726  cx@1=0.590 cy@1=0.293  IoU@0.5=0.427  errCX=3.77 errCY=6.34 errW=7.60 errH=0.69  pkX=0.292 pkY=0.262  ret_g=-0.0236  oth_g=-0.0006  lr=9.91e-05
           modes: co=20.0%  dn=30.0%  normal=50.0%
           layers: L1:0.331 → L2:0.349 → L3:0.364 → L4:0.370 → L5:0.374 → L6:0.373
           L1: 0.00:0.15[48%]  0.05:0.39[39%]  0.10:0.57[10%]  0.15:0.37[2%]  0.20:0.00[0%]  0.25:0.00[0%]  0.30:0.00[0%]  0.35:0.00[0%]  0.40:0.00[0%]  0.45:0.00[0%]
           L2: 0.00:0.12[37%]  0.05:0.32[38%]  0.10:0.54[19%]  0.15:0.54[6%]  0.20:0.17[0%]  0.25:0.00[0%]  0.30:0.00[0%]  0.35:0.00[0%]  0.40:0.00[0%]  0.45:0.00[0%]
           L3: 0.00:0.11[33%]  0.05:0.32[33%]  0.10:0.51[25%]  0.15:0.60[8%]  0.20:0.23[0%]  0.25:0.00[0%]  0.30:0.00[0%]  0.35:0.00[0%]  0.40:0.00[0%]  0.45:0.00[0%]
           L4: 0.00:0.12[32%]  0.05:0.32[32%]  0.10:0.50[26%]  0.15:0.59[9%]  0.20:0.30[1%]  0.25:0.00[0%]  0.30:0.00[0%]  0.3

[4/50]  train=9.8951 (xy=4.109 w=2.549 h=1.537 l1=0.112 gi=0.570)  eval=13.9730  mIoU=0.3756  cx@1=0.572 cy@1=0.320  IoU@0.5=0.420  errCX=3.75 errCY=6.28 errW=7.33 errH=0.69  pkX=0.300 pkY=0.282  ret_g=-0.0324  oth_g=-0.0001  lr=9.84e-05
           modes: co=20.0%  dn=30.0%  normal=50.0%
           layers: L1:0.335 → L2:0.363 → L3:0.368 → L4:0.371 → L5:0.375 → L6:0.376
           L1: 0.00:0.12[41%]  0.05:0.35[39%]  0.10:0.57[16%]  0.15:0.42[3%]  0.20:0.03[0%]  0.25:0.00[0%]  0.30:0.00[0%]  0.35:0.00[0%]  0.40:0.00[0%]  0.45:0.00[0%]
           L2: 0.00:0.10[32%]  0.05:0.30[34%]  0.10:0.50[25%]  0.15:0.63[9%]  0.20:0.22[0%]  0.25:0.00[0%]  0.30:0.00[0%]  0.35:0.00[0%]  0.40:0.00[0%]  0.45:0.00[0%]
           L3: 0.00:0.10[30%]  0.05:0.28[29%]  0.10:0.47[27%]  0.15:0.62[13%]  0.20:0.38[1%]  0.25:0.00[0%]  0.30:0.00[0%]  0.35:0.00[0%]  0.40:0.00[0%]  0.45:0.00[0%]
           L4: 0.00:0.10[29%]  0.05:0.28[28%]  0.10:0.47[29%]  0.15:0.61[14%]  0.20:0.33[1%]  0.25:0.00[0%]  0.30:0.00[0%]  0.

[5/50]  train=9.1484 (xy=3.891 w=2.311 h=1.506 l1=0.092 gi=0.491)  eval=14.1799  mIoU=0.3750  cx@1=0.546 cy@1=0.313  IoU@0.5=0.427  errCX=3.75 errCY=5.91 errW=7.32 errH=0.67  pkX=0.316 pkY=0.297  ret_g=-0.0408  oth_g=0.0000  lr=9.76e-05
           modes: co=20.0%  dn=30.0%  normal=50.0%
           layers: L1:0.331 → L2:0.358 → L3:0.368 → L4:0.373 → L5:0.377 → L6:0.375
           L1: 0.00:0.11[40%]  0.05:0.31[36%]  0.10:0.54[19%]  0.15:0.54[5%]  0.20:0.07[0%]  0.25:0.00[0%]  0.30:0.00[0%]  0.35:0.00[0%]  0.40:0.00[0%]  0.45:0.00[0%]
           L2: 0.00:0.10[31%]  0.05:0.28[29%]  0.10:0.48[25%]  0.15:0.67[14%]  0.20:0.31[1%]  0.25:0.00[0%]  0.30:0.00[0%]  0.35:0.00[0%]  0.40:0.00[0%]  0.45:0.00[0%]
           L3: 0.00:0.10[27%]  0.05:0.25[25%]  0.10:0.45[27%]  0.15:0.63[19%]  0.20:0.43[1%]  0.25:0.00[0%]  0.30:0.00[0%]  0.35:0.00[0%]  0.40:0.00[0%]  0.45:0.00[0%]
           L4: 0.00:0.09[26%]  0.05:0.27[23%]  0.10:0.45[29%]  0.15:0.62[20%]  0.20:0.39[1%]  0.25:0.00[0%]  0.30:0.00[0%]  0.

[6/50] train:   8%|▊         | 60/718 [00:30<04:40,  2.35it/s]